In [19]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

import os
import numpy as np
import random
from tqdm import tqdm
import pickle
from collections import Counter

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
print(torch.cuda.get_device_name(0))

cuda
Tesla T4


In [22]:
LOAD_DIR = '/content/drive/MyDrive/flood_detection_project/data/processed'

with open(f'{LOAD_DIR}/train.pkl', 'rb') as f:
  train_data = pickle.load(f)

with open(f'{LOAD_DIR}/test.pkl', 'rb') as f:
  test_data = pickle.load(f)

with open(f'{LOAD_DIR}/val.pkl', 'rb') as f:
  val_data = pickle.load(f)

print("datasets loaded")
print("train", len(train_data))
print("test", len(test_data))
print("val", len(val_data))

datasets loaded
train 187
test 41
val 40


In [23]:
X, y = train_data[0]
print(X.shape)
print(y)

(4, 516, 544)
1


In [24]:
labels = [y for _, y in train_data]
print(Counter(labels))
print("mean: ", X.mean())
print("std: ", X.std())

Counter({0: 96, 1: 91})
mean:  7.479799e-08
std:  0.9999984


In [25]:
class FloodDataset(Dataset):
  def __init__(self, data):
    self.data = data

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    X, y = self.data[index]
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)
    return X, y


In [44]:
def collate_fn(batch):
  valid_X= []
  valid_y = []

  for X, y in batch:
    if X is not None and X.shape[1] > 0 and X.shape[2] > 0:
      valid_X.append(X)
      valid_y.append(y)

  if len(valid_X) == 0:
    return None, None

  min_h = min(X.shape[1] for X in valid_X)
  min_w = min(X.shape[2] for X in valid_X)

  cropped = []
  for x in valid_X:
    h, w = x.shape[1], x.shape[2]
    sh = (h - min_h) // 2
    sw = (w - min_w) // 2
    cropped.append( x[:, sh:sh+min_h, sw:sw+min_w])

  X_batch = torch.stack(cropped).float()
  y_batch = torch.stack(valid_y).float()
  return X_batch, y_batch

In [45]:
BATHC_SIZE = 2
train_loader = DataLoader(FloodDataset(train_data), batch_size=BATHC_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(FloodDataset(val_data), batch_size=BATHC_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(FloodDataset(test_data), batch_size=BATHC_SIZE, shuffle=False, collate_fn=collate_fn)

print('batches: ', len(train_loader), len(val_loader), len(test_loader))

batches:  94 20 21


In [46]:
Xb, yb = next(iter(train_loader))
print("batch X:",  Xb.shape, "batch y", yb)

batch X: torch.Size([2, 4, 517, 545]) batch y tensor([1., 0.])


In [47]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


In [55]:
def center_crop_to_match(enc_feat, dec_feat):
  _, _, h, w = dec_feat.shape
  enc_h, enc_w = enc_feat.shape[2], enc_feat.shape[3]

  sh = (enc_h - h) // 2
  sw = (enc_w - w) // 2

  return enc_feat[:, :, sh:sh+h, sw:sw+w]

In [62]:
class UNet(nn.Module):
    def __init__(self, in_channels=4):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.up3(b)
        e3c = center_crop_to_match(e3, d3)
        d3 = self.dec3(torch.cat([d3, e3c], dim=1))

        d2 = self.up2(d3)
        e2c = center_crop_to_match(e2, d2)
        d2 = self.dec2(torch.cat([d2, e2c], dim=1))
        d1 = self.up1(d2)
        e1c = center_crop_to_match(e1, d1)
        d1 = self.dec1(torch.cat([d1, e1c], dim=1))

        return self.out(d1)


In [63]:
model = UNet(in_channels=4).to(device)
print("U-Net initialized")


U-Net initialized


In [64]:
def event_prediction(logits):
    return logits.mean(dim=(2, 3))


In [65]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [66]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    count = 0

    for batch in tqdm(loader):
        if batch is None:
            continue

        X, y = batch
        X = X.to(device)
        y = y.to(device).unsqueeze(1)

        optimizer.zero_grad()
        logits = model(X)
        preds = event_prediction(logits)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        count += 1

    return total_loss / max(count, 1)


In [68]:
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    count = 0

    with torch.no_grad():
        for batch in loader:
            if batch is None:
                continue

            X, y = batch
            X = X.to(device)
            y = y.to(device).unsqueeze(1)

            logits = model(X)
            preds = event_prediction(logits)
            loss = criterion(preds, y)

            total_loss += loss.item()
            p = (torch.sigmoid(preds) > 0.5).float()
            correct += (p == y).sum().item()
            total += y.size(0)
            count += 1

    return total_loss / max(count, 1), correct / max(total, 1)


In [71]:
EPOCHS = 30
best_val = float("inf")

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "best_unet.pth")
        print("✓ Best model saved")


100%|██████████| 94/94 [00:37<00:00,  2.49it/s]


Epoch 1/30
Train Loss: 0.6040
Val Loss:   0.5085, Val Acc: 0.7250
✓ Best model saved


100%|██████████| 94/94 [00:38<00:00,  2.44it/s]


Epoch 2/30
Train Loss: 0.6583
Val Loss:   0.5558, Val Acc: 0.7500


100%|██████████| 94/94 [00:37<00:00,  2.48it/s]


...

Epoch 28/30
Train Loss: 0.5548
Val Loss:   0.7149, Val Acc: 0.6750


100%|██████████| 94/94 [00:38<00:00,  2.47it/s]


Epoch 29/30
Train Loss: 0.5677
Val Loss:   0.5042, Val Acc: 0.7250


100%|██████████| 94/94 [00:38<00:00,  2.47it/s]


Epoch 30/30
Train Loss: 0.5781
Val Loss:   0.5518, Val Acc: 0.7250


In [72]:
test_loss, test_acc = evaluate(model, test_loader)
print(test_loss, test_acc)

0.8148965062130065 0.6585365853658537


In [73]:
MODEL_DIR = '/content/drive/MyDrive/flood_detection_project/models'
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, 'unet_30epochs.pth')

torch.save(model.state_dict(), MODEL_PATH)

MODEL_PATH

'/content/drive/MyDrive/flood_detection_project/models/unet_30epochs.pth'

In [74]:
print(os.path.exists(MODEL_PATH))
print(os.path.getsize(MODEL_PATH)/ (1024**2))

True
29.44233226776123
